In [5]:
import os
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d

# Konfiguration
quelle_ordner = ['./5V', './15V', './25V', './35V', './40V']
ziel_ordner = './Messwertebearbeitet'
zielanzahl = 1800  # Anzahl der Punkte in der Ziel-Zeitskala
glatt_fenster = 101  # Fenstergröße für Glättung (ungerade)
glatt_poly = 3      # Polynomgrad

def lade_messdaten(pfad):
    """Lädt die Datei ab der Zeile 'Time,Ampl'."""
    try:
        with open(pfad, 'r', encoding='utf-8') as f:
            zeilen = f.readlines()
        for i, zeile in enumerate(zeilen):
            if zeile.strip().startswith("Time"):
                header_index = i
                break
        else:
            raise ValueError("Keine 'Time'-Zeile gefunden")

        df = pd.read_csv(pfad, skiprows=header_index)
        df.columns = ['Time', 'Amplitude']
        df = df.dropna()
        df['Time'] = pd.to_numeric(df['Time'], errors='coerce')
        df['Amplitude'] = pd.to_numeric(df['Amplitude'], errors='coerce')
        df = df.dropna()
        return df
    except Exception as e:
        print(f"❌ Fehler bei Datei {pfad}: {e}")
        return None

def verarbeite_daten(dfs, gemeinsame_zeitachse):
    """Interpoliert, glättet und normalisiert alle Messreihen auf eine gemeinsame Zeitskala."""
    bearbeitet = {}
    for (ordner, datei), df in dfs.items():
        try:
            # Zeit auf 0 setzen
            df['Time'] = df['Time'] - df['Time'].iloc[0]

            # Interpolation auf die gemeinsame Zeitskala
            f = interp1d(df['Time'].values, df['Amplitude'].values, kind='linear', bounds_error=False, fill_value='extrapolate')
            amplituden = f(gemeinsame_zeitachse)

            # Negative Werte auf 0 setzen
            amplituden = np.clip(amplituden, 0, None)

            # Glättung anwenden (nur wenn genug Punkte)
            if len(amplituden) >= glatt_fenster:
                amplituden = savgol_filter(amplituden, glatt_fenster, glatt_poly)

            bearbeitet[(ordner, datei)] = pd.DataFrame({
                'Time': gemeinsame_zeitachse,
                'Amplitude': amplituden
            })
        except Exception as e:
            print(f"⚠️ Fehler beim Verarbeiten von {ordner}/{datei}: {e}")
    return bearbeitet

def speichere_bearbeitet(bearbeitet):
    for (ordner, datei), df in bearbeitet.items():
        zielpfad_ordner = os.path.join(ziel_ordner, os.path.basename(ordner))
        os.makedirs(zielpfad_ordner, exist_ok=True)
        zielpfad = os.path.join(zielpfad_ordner, datei)
        df.to_csv(zielpfad, index=False)

def main():
    daten = {}
    endzeiten = []

    # Alle Dateien einlesen
    for ordner in quelle_ordner:
        if not os.path.isdir(ordner):
            print(f"⚠️ Ordner {ordner} nicht gefunden.")
            continue
        for datei in os.listdir(ordner):
            if datei.endswith('.txt'):
                pfad = os.path.join(ordner, datei)
                df = lade_messdaten(pfad)
                if df is not None:
                    df['Time'] = df['Time'] - df['Time'].iloc[0]
                    daten[(ordner, datei)] = df
                    endzeiten.append(df['Time'].iloc[-1])

    if not daten:
        print("❌ Keine gültigen Dateien gefunden.")
        return

    # Gemeinsame maximale Zeitdauer ermitteln
    t_ende = min(endzeiten)
    gemeinsame_zeitachse = np.linspace(0, t_ende, zielanzahl)

    # Verarbeitung
    bearbeitet = verarbeite_daten(daten, gemeinsame_zeitachse)
    speichere_bearbeitet(bearbeitet)
    print(f"✅ Alle bearbeiteten Dateien wurden in '{ziel_ordner}' gespeichert.")

if __name__ == "__main__":
    main()


✅ Alle bearbeiteten Dateien wurden in './Messwertebearbeitet' gespeichert.
